# 08 — Esparsidade, viés e encolhimento

**DATA SCIENCE KNOWLEDGE BASE**  
**Área:** `09_recommender_systems` — Sistemas de Recomendação  
**Capítulo:** 08 de 16  

> Material didático autoral revisado a partir dos notebooks de estudo. Execute as células em sequência, de cima para baixo, em um kernel reiniciado. Consulte as fontes históricas somente para estudo, preservando sua proveniência.

## Objetivos de aprendizagem
- Calcular coavaliações
- Corrigir similaridade por quantidade de evidências

## Materiais complementares

- [Guia Técnico](reference_materials/08_esparsidade_regularizacao.pdf)
- [Cheat Sheet](cheat_sheets/cheat_sheet_08_esparsidade_regularizacao.pdf)

**Navegação:** os links são relativos à pasta deste notebook. A versão editorial desses PDFs será revalidada após a consolidação científica dos notebooks.

## Fundamentação conceitual
Para dois usuários, a similaridade de Pearson calculada somente sobre itens coavaliados é instável com pouquíssimo suporte. Um encolhimento simples é $s^*=s\,n/(n+\lambda)$, onde $n$ é o número de itens compartilhados.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.metrics.pairwise import cosine_similarity
DATA = Path('datasets')
if not (DATA / 'ratings.csv').exists():
    raise FileNotFoundError('Abra o JupyterLab na pasta da Disciplina 09 (onde existe datasets/).')
ratings = pd.read_csv(DATA / 'ratings.csv')
movies = pd.read_csv(DATA / 'movies.csv')
print('Avaliações:', len(ratings), '| Filmes:', len(movies), '| Usuários:', ratings.userId.nunique())

Avaliações: 100836 | Filmes: 9742 | Usuários: 610


## Pearson em coavaliações

In [2]:
def pearson_shrunk(df,a,b,shrink=10):
    both=df.loc[[a,b]].T.dropna()
    n=len(both)
    if n<2 or both.iloc[:,0].std()==0 or both.iloc[:,1].std()==0: return 0.0,n
    corr=both.iloc[:,0].corr(both.iloc[:,1]); return float(corr*n/(n+shrink)),n
M=ratings.pivot(index='userId',columns='movieId',values='rating')
for user in [2,3,4,5]:print('1 vs',user,pearson_shrunk(M,1,user))

1 vs 2 (0.0, 2)
1 vs 3 (0.03286658821837284, 7)
1 vs 4 (0.17016788169973027, 45)
1 vs 5 (0.15190162358674028, 13)


**Interpretação.** Repare que a confiança depende tanto do coeficiente quanto da quantidade de itens comuns.

## Efeito da regularização

In [3]:
print('Comparação de lambda:',[pearson_shrunk(M,1,4,s) for s in [0,5,20,50]])

Comparação de lambda: [(0.20798296652189258, 45), (0.1871846698697033, 45), (0.14398820759207948, 45), (0.09851824729984385, 45)]


**Interpretação.** Encolhimento reduz intensidade de correlações baseadas em poucos dados.

## Aprofundamento e aplicação

**Ponto de atenção deste capítulo:** shrinkage. Este complemento não substitui as seções e exemplos anteriores.

**Verificação conceitual.** Explique, com suas palavras, em qual etapa do fluxo de trabalho o tema **shrinkage** pode produzir conclusões enganosas. Indique um indicador observável, um teste mínimo e uma limitação do procedimento.

**Exercício de extensão.** Reproduza o experimento central deste notebook alterando **um parâmetro por vez**. Compare com uma linha de base simples, reporte a variação da métrica pertinente e justifique o resultado. Quando houver treino/teste, qualquer etapa que aprenda parâmetros deve usar apenas o treino.

**Checklist de conclusão:** (1) entradas identificadas; (2) sementes registradas; (3) saídas interpretadas; (4) premissas explicitadas; (5) dados originais não sobrescritos.


### Laboratório adicional — Shrinkage de similaridade para poucos coavaliadores

Experimento independente do estado das células anteriores. Use-o como verificação e explique seus limites; resultados em conjuntos sintéticos não são estimativas de desempenho real.

In [4]:
import numpy as np
for n in (1,3,20,100):
 similaridade=.9
 regularizada=similaridade*n/(n+10)
 print('coavaliações:',n,'similaridade regularizada:',round(regularizada,3))

coavaliações: 1 similaridade regularizada: 0.082
coavaliações: 3 similaridade regularizada: 0.208
coavaliações: 20 similaridade regularizada: 0.6
coavaliações: 100 similaridade regularizada: 0.818


### Exercício resolvido — protocolo mínimo de validação

**Enunciado.** Proponha uma comparação para **shrinkage** sem contaminar a avaliação.

**Solução orientadora.** Defina uma hipótese falsificável, escolha um baseline simples, fixe uma divisão adequada ao problema, altere uma única decisão metodológica e compare métricas no conjunto reservado. Não transforme exemplos sintéticos em conclusões sobre populações reais. Documente a configuração e os casos de falha. Quando o tema não envolver predição, substitua o baseline por uma implementação de referência e compare correção, cobertura e custo.


## Análise crítica — Coavaliação, correlação e shrinkage

Pearson sobre itens coavaliados é **indefinido** com menos de dois pares ou variância nula. O notebook retorna zero nessas situações por convenção operacional; isso não significa independência. O shrinkage `s* = s × n/(n+λ)` reduz estimativas sustentadas por poucos pares. Escolha `λ` na validação.

**Origem:** aprofundamento técnico autoral fundamentado no experimento deste notebook. Não é transcrição dos materiais históricos.

In [5]:
# Demonstrar monotonicidade do fator de encolhimento para suporte fixo
n = 3
fatores = {l: n/(n+l) for l in [0,5,20,50]}
print('Suporte =', n, 'fatores de shrinkage:', fatores)
assert all(0 <= f <= 1 for f in fatores.values())
assert list(fatores.values()) == sorted(fatores.values(), reverse=True)

Suporte = 3 fatores de shrinkage: {0: 1.0, 5: 0.375, 20: 0.13043478260869565, 50: 0.05660377358490566}


### Critério de leitura

Interprete os resultados como diagnóstico didático. Para comparar modelos de recomendação, defina previamente população, itens elegíveis, janela temporal, sinal de relevância, baseline e métrica; mantenha treino, validação e teste separados.

**Atividade:** registre o efeito de uma mudança de parâmetro e explique uma ameaça à validade do experimento.

## Limitações e boas práticas
Encolhimento é uma heurística; parâmetros devem ser escolhidos sem consultar o teste.

## Exercícios
Por que o resultado deve ser zero para menos de duas coavaliações?

## Soluções comentadas
Não há pares suficientes para medir correlação, e a variância pode ser nula.

## Síntese
Reveja os resultados e compare métodos usando um protocolo de dados e métricas consistente. Não trate avaliações ausentes como avaliações negativas.

### Exportação de resultados

A próxima célula grava os resultados do capítulo em `outputs/`.


In [6]:
# EP_OUTPUTS_09_V1
# Exportação após os experimentos. Executar este notebook desde o início (Run All).
# Arquivos com prefixo do capítulo evitam colisões com outras saídas.
from pathlib import Path
out = Path('outputs')
out.mkdir(parents=True, exist_ok=True)
pd.DataFrame([{"usuario_a": 1, "usuario_b": 4, "lambda": l,
               "similaridade_regularizada": pearson_shrunk(M, 1, 4, l)[0],
               "coavaliacoes": pearson_shrunk(M, 1, 4, l)[1]}
              for l in [0, 5, 20, 50]]).to_csv(
    out / "08_efeito_regularizacao.csv", index=False)
print('Outputs do capítulo 08 salvos em:', out.resolve())


Outputs do capítulo 08 salvos em: /run/media/eduardo/Backup/Faculdades/PUC - CIENCIA DE DADOS/PUC/02_Disciplinas/09_recommender_systems/outputs
